# 02. Exploratory Data Analysis: Diet Recommendations Target Leakage Proof

**Dataset:** `diet_recommendations_dataset.csv` (1,000 rows, 20 columns)  
**Goal:** Prove conclusively that `Diet_Recommendation` is 100% deterministically determined by `Disease_Type`.

---
### The Problem:
Training a classifier to predict `Diet_Recommendation` yields 100% test accuracy. In peer reviews or portfolios, this looks like great ML, but is in fact trivial leakage of a synthetic generation rule.


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("../data/raw/diet_recommendations_dataset.csv")
print(f"Dataset Shape: {df.shape}")
df[["Patient_ID", "Disease_Type", "Diet_Recommendation"]].head(10)

### Deterministic Cross-Tabulation Proof
Let's construct the contingency table between `Disease_Type` and `Diet_Recommendation`:


In [ ]:
crosstab = pd.crosstab(df["Disease_Type"].fillna("Missing / None"), df["Diet_Recommendation"])
print("Contingency Table (Disease_Type vs Diet_Recommendation):")
print(crosstab)

### Notice the 100% Deterministic Rule:
- `Disease_Type == 'Diabetes'` $\to$ 100% `Low_Carb` (258/258, 0 other)
- `Disease_Type == 'Hypertension'` $\to$ 100% `Low_Sodium` (316/316, 0 other)
- `Disease_Type == 'Obesity'` $\to$ 100% `Balanced` (222/222, 0 other)
- `Disease_Type == NaN` $\to$ 100% `Balanced` (204/204, 0 other)

Let's confirm cross-validated accuracy using only `Disease_Type`:


In [ ]:
X = pd.get_dummies(df[["Disease_Type"]].fillna("None"), drop_first=False)
y = df["Diet_Recommendation"]

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
scores = cross_val_score(clf, X, y, cv=5)
print(f"5-Fold CV Accuracy: {scores.mean() * 100:.2f}% (+/- {scores.std() * 100:.2f}%)")

### Conclusion and Architectural Impact
- The label `Diet_Recommendation` is a synthetic 1-to-1 mapping of `Disease_Type`.
- **Decision (DECISIONS.md):** We do NOT train an ML model on this label. Instead, personalized diet recommendations must be generated by an evidence-based, transparent clinical rule engine (DASH, ADA, AHA guidelines) and mathematical linear programming.
- We utilize this dataset exclusively for realistic vocabulary (allergies, restrictions, cuisines) and demo user profiles.
